In [ ]:
import numpy as np
import copy

## Bagging

In [3]:
class Bagging:
  def __init__(self, base_learner, n_learners):
    self.learners = [copy.clone(base_learner) for _ in range(n_learners)]
    
  def fit(self, X, y):
    for learner in self.learners:
      examples = np.random.choice(
        np.arange(len(X)), int(len(X)), replace=True
      )
      learner.fit(X.iloc[examples, :], y.iloc[examples])
      
  def predict(self, X):
    preds = [learner.predict(X) for learner in self.learners]
    return np.array(preds).mean(axis=0)

## Boosting

In [4]:
class GradientBoosting:
  def __init__(self, base_learner, n_learners, learning_rate):
    self.learners = [copy.clone(base_learner) for _ in range(n_learners)]
    self.lr = learning_rate
  
  def fit(self, X, y):
    residual = y.copy()
    for learner in self.learners:
      learner.fit(X, residual)
      residual -= self.lr * learner.predict(X)
      
  def predict(self, X):
    preds = [learner.predict(X) for learner in self.learners]
    return np.array(preds).sum(axis=0) * self.lr

## 5.4. Stacking

In [6]:
from autogluon.tabular import TabularPredictor, TabularDataset
import pandas as pd

train = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/Inc/train.csv')
label = 'occupation' # Column name

predict = TabularPredictor(label=label).fit(
  train, num_stack_levels=1, num_bag_folds=5
)

Loaded data from: https://autogluon.s3.amazonaws.com/datasets/Inc/train.csv | Columns = 15 / 15 | Rows = 39073 -> 39073
No path specified. Models will be saved in: "AutogluonModels\ag-20250214_194003"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.9.21
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22631
CPU Count:          12
Memory Avail:       0.42 GB / 7.85 GB (5.4%)
Disk Space Avail:   61.42 GB / 952.59 GB (6.4%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'  

In [7]:
test_data = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/Inc/test.csv')
predictions = predict.predict(test_data)

Loaded data from: https://autogluon.s3.amazonaws.com/datasets/Inc/test.csv | Columns = 15 / 15 | Rows = 9769 -> 9769


In [8]:
predictions

0           Other-service
1         Farming-fishing
2         Exec-managerial
3           Other-service
4           Other-service
              ...        
9764         Adm-clerical
9765         Craft-repair
9766     Transport-moving
9767        Other-service
9768         Adm-clerical
Name: occupation, Length: 9769, dtype: object